# Lab: Proposition 99 With Transparent SCM

[Website](https://defenceeconomist.github.io/qedlabs/labs/synthetic-control-proposition-99-lab.html)

Use the R kernel. Keep the supplied `data/` folder beside this notebook. Run cells in order after installing the documented R environment. Data loading is entirely local.

## How To Use This Page

Use this page as the first full SCM application lab.

- Keep the [Synthetic Control](https://defenceeconomist.github.io/qedlabs/notes/scm/synthetic-control.html) overview open in another tab.
- Use the [California Proposition 99 notes](https://defenceeconomist.github.io/qedlabs/notes/scm/california-tobacco-synthetic-control-notes.html) when you want the paper-level rationale behind each design step.
- Do not rush to the post-treatment plot. The donor pool, predictors, and pre-treatment fit are the actual argument.
- End with a design judgment about whether California’s synthetic comparison looks credible enough to interpret.

The code is shown but not executed when the site is rendered. That keeps the page readable while preserving a runnable workflow for teaching and self-study.

## Training Goal

Turn the canonical Proposition 99 application into a repeatable SCM workflow using explicit matrices and base-R optimisation:

1.  define the treated unit and intervention date
2.  build a donor pool that tries to stay plausibly untreated
3.  choose predictors and lagged outcomes that discipline pre-treatment fit
4.  inspect donor weights, balance, path fit, and placebo-style comparisons
5.  explain why this is a design-and-diagnostics workflow rather than just a nice graph

## Dataset At A Glance

This lab uses a checksummed snapshot of the `smoking` state-year panel tied to California’s tobacco-control program. The data are bundled, so the package that originally distributed them is not required at runtime.

- Treated unit: California
- Intervention anchor: `1988` in the teaching implementation, so `1989` onward is interpreted as the post-treatment period
- Outcome: per-capita cigarette sales (`cigsale`)
- Core predictors: income, cigarette price, beer consumption, age structure, and lagged smoking outcomes
- Main value: this is the cleanest benchmark case for teaching a full end-to-end SCM workflow while keeping the optimisation visible

## What To Hand Back

By the end of the lab, you should be able to report:

- which states were excluded before estimation and why
- which predictors anchor the fit exercise
- which donor states receive positive or near-positive weight
- whether California’s pre-treatment fit is strong enough to make the `1989` onward gap meaningful
- whether the placebo diagnostics reinforce the result or only partially reassure you

## Step 1: Load Packages And Inspect The Panel

In [ ]:
options(repr.plot.width = 8, repr.plot.height = 4.8)
data_helpers <- c("data/load-data.R", "../data/load-data.R", "docs/labs/data/load-data.R")
data_helpers <- data_helpers[file.exists(data_helpers)]
if (!length(data_helpers)) stop("Extract the complete lab ZIP, including its data folder, before running.")
source(data_helpers[[1]])

In [ ]:
options(repr.plot.width = 8, repr.plot.height = 4.8)
required_packages <- c(
  "dplyr",
  "ggplot2",
  "tibble",
  "tidyr"
)

missing_packages <- required_packages[!vapply(
  required_packages,
  requireNamespace,
  logical(1),
  quietly = TRUE
)]

if (length(missing_packages) > 0) {
  stop("Install the documented R environment first; missing: ", paste(missing_packages, collapse=", "), call.=FALSE)
}

invisible(lapply(required_packages, library, character.only = TRUE))

smoking <- qed_data("smoking")

smoking |>
  glimpse()

Checkpoint:

- The data are already in tidy long format.
- Every row is a state-year observation, which makes the construction of predictor and outcome matrices easy to audit.

## Step 2: Define The Intervention And Donor Exclusions

Following the canonical Proposition 99 design, exclude states that adopted large tobacco-control programs during the post period, states with very large cigarette-tax increases, and the District of Columbia.

In [ ]:
options(repr.plot.width = 8, repr.plot.height = 4.8)
treated_state <- "California"
intervention_year <- 1988

excluded_states <- c(
  "Alaska",
  "Hawaii",
  "Maryland",
  "Massachusetts",
  "Michigan",
  "New Jersey",
  "New York",
  "Washington",
  "District of Columbia"
)

analysis_data <- smoking |>
  filter(year <= 2000) |>
  filter(!state %in% excluded_states)

analysis_data |>
  count(state, sort = TRUE)

What to discuss:

- donor-pool construction is part of identification, not housekeeping
- the point is not to find the biggest donor pool, but the most defensible untreated comparison set
- if contamination remains plausible even after exclusions, that is a design limitation you should say out loud

## Step 3: Inspect The Raw California Trend

In [ ]:
options(repr.plot.width = 8, repr.plot.height = 5.5)
analysis_data |>
  mutate(group = if_else(state == treated_state, "California", "Donor pool")) |>
  group_by(group, year) |>
  summarise(cigsale = mean(cigsale, na.rm = TRUE), .groups = "drop") |>
  ggplot(aes(x = year, y = cigsale, color = group)) +
  geom_line(linewidth = 1.1) +
  geom_vline(xintercept = intervention_year, linetype = 2, color = "gray40") +
  labs(
    x = NULL,
    y = "Per-capita cigarette sales",
    color = NULL
  ) +
  theme_minimal(base_size = 12)

Checkpoint:

- A simple donor-pool average is not the SCM counterfactual.
- This first plot is only a benchmark for why a weighted synthetic comparison is needed.

## Step 4: Build The Synthetic-Control Design

In [ ]:
options(repr.plot.width = 8, repr.plot.height = 4.8)
pre_years <- 1970:1988
plot_years <- 1970:2000
donor_states <- setdiff(unique(analysis_data$state), treated_state)

unit_predictors <- function(state_name, data = analysis_data) {
  unit_data <- data |>
    filter(state == state_name)

  c(
    lnincome = mean(unit_data$lnincome[unit_data$year %in% 1980:1988], na.rm = TRUE),
    retprice = mean(unit_data$retprice[unit_data$year %in% 1980:1988], na.rm = TRUE),
    age15to24 = mean(unit_data$age15to24[unit_data$year %in% 1980:1988], na.rm = TRUE),
    beer = mean(unit_data$beer[unit_data$year %in% 1984:1988], na.rm = TRUE),
    cigsale_1975 = unit_data$cigsale[match(1975, unit_data$year)],
    cigsale_1980 = unit_data$cigsale[match(1980, unit_data$year)],
    cigsale_1988 = unit_data$cigsale[match(1988, unit_data$year)]
  )
}

unit_outcomes <- function(state_name, years, data = analysis_data) {
  unit_data <- data |>
    filter(state == state_name)
  unit_data$cigsale[match(years, unit_data$year)]
}

simplex_weights <- function(theta) c(theta, 1 - sum(theta))

fit_scm <- function(target_state, comparison_states, data = analysis_data) {
  X1 <- unit_predictors(target_state, data)
  X0 <- vapply(comparison_states, unit_predictors, numeric(length(X1)), data = data)
  Z1 <- unit_outcomes(target_state, pre_years, data)
  Z0 <- vapply(
    comparison_states,
    unit_outcomes,
    numeric(length(pre_years)),
    years = pre_years,
    data = data
  )

  design_target <- c(X1, Z1)
  design_donors <- rbind(X0, Z0)
  row_scale <- apply(cbind(design_target, design_donors), 1, sd)
  row_scale[!is.finite(row_scale) | row_scale == 0] <- 1
  scaled_target <- design_target / row_scale
  scaled_donors <- design_donors / row_scale

  loss <- function(theta) {
    weights <- simplex_weights(theta)
    mean((scaled_target - as.numeric(scaled_donors %*% weights))^2)
  }

  gradient <- function(theta) {
    weights <- simplex_weights(theta)
    residual <- scaled_target - as.numeric(scaled_donors %*% weights)
    donor_differences <- scaled_donors[, -ncol(scaled_donors), drop = FALSE] -
      scaled_donors[, ncol(scaled_donors)]
    -2 * colMeans(donor_differences * residual)
  }

  constraint_matrix <- rbind(
    diag(length(comparison_states) - 1),
    rep(-1, length(comparison_states) - 1)
  )
  constraint_boundary <- c(rep(0, length(comparison_states) - 1), -1)

  fit <- constrOptim(
    theta = rep(1 / length(comparison_states), length(comparison_states) - 1),
    f = loss,
    grad = gradient,
    ui = constraint_matrix,
    ci = constraint_boundary,
    method = "BFGS",
    control = list(maxit = 5000, reltol = 1e-11)
  )
  weights <- simplex_weights(fit$par)

  stopifnot(all(weights >= 0), abs(sum(weights) - 1) < 1e-10)
  list(
    weights = weights,
    comparison_states = comparison_states,
    predictors = X1,
    donor_predictors = X0,
    convergence = fit$convergence,
    objective = fit$value
  )
}

california_fit <- fit_scm(treated_state, donor_states)

The helper keeps the design work visible. Base R’s `constrOptim()` constrains donor weights to be non-negative and sum to one while minimizing standardized predictor and pre-treatment outcome mismatch.

## Step 5: Inspect Balance Before You Look At Effects

In [ ]:
options(repr.plot.width = 8, repr.plot.height = 4.8)
balance_tbl <- tibble(
  predictor = names(california_fit$predictors),
  California = california_fit$predictors,
  synthetic = as.numeric(california_fit$donor_predictors %*% california_fit$weights),
  donor_average = rowMeans(california_fit$donor_predictors)
)

balance_tbl

What to look for:

- the synthetic California column should sit close to observed California on the key predictors
- the donor-pool average is the benchmark you are trying to beat
- weak lagged-outcome balance is a warning sign because those lags are carrying a lot of the design discipline

## Step 6: Inspect Donor Weights

In [ ]:
options(repr.plot.width = 8, repr.plot.height = 4.8)
weights_tbl <- tibble(
  state = california_fit$comparison_states,
  weight = california_fit$weights
) |>
  arrange(desc(weight))

weights_tbl

Checkpoint:

- donor weights are part of the result, not an appendix
- if the synthetic control is built from a small set of plausible states, the counterfactual becomes easier to inspect and defend

## Step 7: Plot Observed And Synthetic California

In [ ]:
options(repr.plot.width = 8, repr.plot.height = 5.7)
treated_path <- unit_outcomes(treated_state, plot_years)
donor_paths <- vapply(
  donor_states,
  unit_outcomes,
  numeric(length(plot_years)),
  years = plot_years
)

gap_tbl <- tibble(
  year = plot_years,
  observed = treated_path,
  synthetic = as.numeric(donor_paths %*% california_fit$weights)
) |>
  mutate(gap = observed - synthetic)

gap_tbl |>
  select(year, observed, synthetic) |>
  pivot_longer(-year, names_to = "series", values_to = "cigsale") |>
  ggplot(aes(year, cigsale, colour = series)) +
  geom_line(linewidth = 1) +
  geom_vline(xintercept = intervention_year, linetype = 2, colour = "gray40") +
  labs(x = NULL, y = "Per-capita cigarette sales", colour = NULL) +
  theme_minimal(base_size = 12)

In [ ]:
options(repr.plot.width = 8, repr.plot.height = 5.7)
ggplot(gap_tbl, aes(year, gap)) +
  geom_hline(yintercept = 0, linetype = 3, colour = "gray50") +
  geom_vline(xintercept = intervention_year, linetype = 2, colour = "gray40") +
  geom_line(linewidth = 1) +
  labs(x = NULL, y = "California minus synthetic California") +
  theme_minimal(base_size = 12)

What to discuss:

- the trend plot asks whether California and synthetic California line up closely through `1988`
- the difference plot asks whether the post-treatment divergence is both visible and treatment-timed
- a dramatic post-treatment gap is not enough if the pre-treatment fit is loose

## Step 8: Make The Gap Series Inspectable

In [ ]:
options(repr.plot.width = 8, repr.plot.height = 4.8)
gap_tbl

This is a useful discipline step. Never rely only on the plot when you can inspect the actual treated-versus-synthetic series directly.

## Step 9: Add Placebo-By-Unit Diagnostics

In [ ]:
options(repr.plot.width = 8, repr.plot.height = 6)
placebo_paths <- lapply(donor_states, function(placebo_state) {
  placebo_donors <- setdiff(donor_states, placebo_state)
  placebo_fit <- fit_scm(placebo_state, placebo_donors)
  observed <- unit_outcomes(placebo_state, plot_years)
  donors <- vapply(
    placebo_donors,
    unit_outcomes,
    numeric(length(plot_years)),
    years = plot_years
  )
  tibble(
    state = placebo_state,
    year = plot_years,
    gap = observed - as.numeric(donors %*% placebo_fit$weights)
  )
})

placebo_gap_tbl <- bind_rows(placebo_paths)

ggplot(placebo_gap_tbl, aes(year, gap, group = state)) +
  geom_line(colour = "gray75", linewidth = 0.35) +
  geom_line(
    data = gap_tbl,
    aes(year, gap),
    inherit.aes = FALSE,
    colour = "firebrick",
    linewidth = 1.1
  ) +
  geom_hline(yintercept = 0, colour = "gray50") +
  geom_vline(xintercept = intervention_year, linetype = 2, colour = "gray40") +
  labs(x = NULL, y = "Treated minus synthetic") +
  theme_minimal(base_size = 12)

In [ ]:
options(repr.plot.width = 8, repr.plot.height = 4.8)
mspe_tbl <- bind_rows(
  placebo_gap_tbl,
  gap_tbl |>
    transmute(state = treated_state, year, gap)
) |>
  group_by(state) |>
  summarise(
    pre_mspe = mean(gap[year <= intervention_year]^2),
    post_mspe = mean(gap[year > intervention_year]^2),
    mspe_ratio = post_mspe / pre_mspe,
    .groups = "drop"
  ) |>
  arrange(desc(mspe_ratio))

mspe_tbl

How to read this:

- placebos ask whether California’s post-treatment divergence looks unusually large relative to donor states reassigned as if they were treated
- this is diagnostic reassurance, not a substitute for a credible donor pool
- if many placebo units fit terribly before treatment, interpret the ranking with caution

## Step 10: Write The Design Judgment

Use the outputs above to answer:

1.  Why is California better suited to SCM than a simple treated-versus-rest-of-US comparison?
2.  Which donor states actually build synthetic California?
3.  Is the pre-treatment fit strong enough that the `1989` onward gap is worth interpreting?
4.  Do the placebo diagnostics strengthen the case, or do they mainly show how much the result depends on fit quality?

## Next Step

Move next to the [augmented SCM lab](https://defenceeconomist.github.io/qedlabs/labs/synthetic-control-augmentation-lab.html), where the main question shifts from “how do I run classical SCM?” to “what should I do when classical fit is visibly weak?”